# Week 1: Data Collection & Exploratory Data Analysis (EDA)

**Project:** AI-Based Resume Screening and Candidate Shortlisting System  
**Program:** B.Tech Information Technology Project (Under Guidance of HCL Tech)  
**Milestone:** Week 1 of 9  
**Status:** Completed  

---

## 1. Objectives and Scope
1. Synthesize a realistic dataset of 1,000+ candidate resumes across 5 technical job profiles.
2. Introduce controlled data imperfections (missing values, duplicates, outliers) to validate downstream cleaning pipelines.
3. Perform preliminary Exploratory Data Analysis (EDA) on candidate qualifications, experience, and shortlisting outcomes.
4. Establish baseline data schema and descriptive statistics for feature engineering.

---

## 2. Data Collection and Generation

In [ ]:
import os
import sys
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# Set plotting aesthetics
plt.style.use('dark_background')
sns.set_palette('husl')

print('Core libraries imported successfully.')

In [ ]:
# Load generator function from src/generate_data.py
sys.path.append(os.path.join('..', 'src'))
from generate_data import make_dataset

df_raw = make_dataset(n=1000)

# Ensure output directory exists
os.makedirs('../data', exist_ok=True)
df_raw.to_csv('../data/raw_resumes.csv', index=False)

print(f'Raw dataset generated and saved to ../data/raw_resumes.csv')
print(f'Dataset Dimensions: {df_raw.shape[0]} rows, {df_raw.shape[1]} columns')

### Design Rationale for Synthetic Dataset

| Feature / Parameter | Configuration | Engineering Rationale |
|---|---|---|
| Sample Size | 1,020 rows | Appropriate sample size for cross-validation and feature scaling tests. |
| Target Roles | 5 profiles | Covers common recruitment tracks: Data Scientist, ML Engineer, Python Dev, Web Dev, Analyst. |
| Missing Values | ~3% on numeric columns | Simulates candidate omissions in online application forms. |
| Duplicate Records | ~2% | Emulates duplicate application submissions across job portals. |
| Outlier Records | 8 rows | Tests data preprocessing boundary limits (e.g., extreme experience or CGPA values). |
| Label Logic | Multi-attribute scoring | Reflects realistic qualification benchmarks with controlled stochastic noise. |

## 3. Dataset Overview and Schema Inspection

In [ ]:
df = pd.read_csv('../data/raw_resumes.csv')
print('=== DATASET DIMENSIONS ===')
print(f'Total Rows    : {len(df):,}')
print(f'Total Columns : {len(df.columns)}')
print('\n=== FIRST 5 RECORDS ===')
df.head()

In [ ]:
# Column schema, data types, and null value analysis
print('=== COLUMN SCHEMA & INTEGRITY ===')
info_df = pd.DataFrame({
    'Column': df.columns,
    'Dtype': [str(t) for t in df.dtypes],
    'Non-Null Count': df.notnull().sum().values,
    'Null Count': df.isnull().sum().values,
    'Null Ratio (%)': [f"{round(df[c].isnull().mean()*100, 2)}%" for c in df.columns]
})
print(info_df.to_string(index=False))

In [ ]:
print('=== DESCRIPTIVE STATISTICS (NUMERICAL) ===')
df.describe().round(2)

## 4. Data Quality and Anomaly Inspection

In [ ]:
total_cells = df.shape[0] * df.shape[1]
missing_total = df.isnull().sum().sum()
duplicates = df.duplicated().sum()

print('====================================================')
print('          DATA INTEGRITY AUDIT REPORT')
print('====================================================')
print(f'Total Records       : {len(df):,}')
print(f'Total Features      : {len(df.columns)}')
print(f'Total Missing Cells : {missing_total} ({round(missing_total/total_cells*100, 2)}% of matrix)')
print(f'Duplicate Rows      : {duplicates}')
print(f'Shortlisted (1)     : {df["Shortlisted"].sum()} ({round(df["Shortlisted"].mean()*100, 1)}%)')
print(f'Rejected (0)        : {(df["Shortlisted"]==0).sum()} ({round((df["Shortlisted"]==0).mean()*100, 1)}%)')
print('====================================================')

In [ ]:
# Outlier detection via IQR method
print('=== OUTLIER AUDIT (IQR METHOD: 1.5x IQR) ===')
for col in ['Experience_Years', 'CGPA', 'Projects_Count']:
    q1  = df[col].quantile(0.25)
    q3  = df[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    outliers = df[(df[col] < lower) | (df[col] > upper)][col]
    print(f'{col}:')
    print(f'  Valid Boundary : [{round(lower, 2)}, {round(upper, 2)}]')
    print(f'  Outlier Count  : {len(outliers)}')
    if len(outliers) > 0:
        print(f'  Sample Values  : {sorted(outliers.tolist())[:8]}\n')

## 5. Exploratory Data Analysis & Visualizations

In [ ]:
# Figure 1: Target Variable Distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Figure 1: Target Variable Distribution (Shortlisted vs Rejected)', fontsize=13, fontweight='bold', color='white')

counts = df['Shortlisted'].value_counts()
colors = ['#4ade80', '#f87171']

axes[0].pie(
    counts.values, labels=['Shortlisted', 'Rejected'],
    autopct='%1.1f%%', colors=colors, startangle=90,
    textprops={'color': 'white', 'fontsize': 11}
)
axes[0].set_title('Class Ratio', color='white')

bars = axes[1].bar(['Rejected (0)', 'Shortlisted (1)'], counts.sort_index().values, color=colors[::-1], width=0.5)
for bar in bars:
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                 str(int(bar.get_height())), ha='center', va='bottom', color='white', fontweight='bold')
axes[1].set_ylabel('Candidate Count', color='white')
axes[1].set_title('Absolute Frequency', color='white')
axes[1].tick_params(colors='white')

plt.tight_layout()
plt.show()

In [ ]:
# Figure 2: Numerical Feature Distributions
num_cols = ['Experience_Years', 'CGPA', 'Projects_Count']
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Figure 2: Distribution of Numerical Predictors', fontsize=13, fontweight='bold', color='white')
palette = ['#6366f1', '#8b5cf6', '#06b6d4']

for ax, col, color in zip(axes, num_cols, palette):
    data = df[col].dropna()
    ax.hist(data, bins=25, color=color, alpha=0.85)
    ax.axvline(data.mean(), color='white', linestyle='--', linewidth=1.2, label=f'Mean: {data.mean():.2f}')
    ax.axvline(data.median(), color='#facc15', linestyle=':', linewidth=1.2, label=f'Median: {data.median():.2f}')
    ax.set_title(col, color='white')
    ax.set_xlabel('Value', color='#94a3b8')
    ax.set_ylabel('Frequency', color='#94a3b8')
    ax.tick_params(colors='white')
    ax.legend(fontsize=8, labelcolor='white')

plt.tight_layout()
plt.show()

In [ ]:
# Figure 3: Job Role and Education Distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
fig.suptitle('Figure 3: Categorical Profile Distribution', fontsize=13, fontweight='bold', color='white')

role_counts = df['Job_Role'].value_counts()
axes[0].barh(role_counts.index, role_counts.values, color='#6366f1')
for i, v in enumerate(role_counts.values):
    axes[0].text(v + 2, i, str(v), va='center', color='white', fontsize=10)
axes[0].set_title('Candidate Breakdown by Role', color='white')
axes[0].tick_params(colors='white')
axes[0].set_xlabel('Count', color='#94a3b8')

edu_counts = df['Education'].value_counts()
axes[1].bar(edu_counts.index, edu_counts.values, color='#8b5cf6', width=0.5)
for i, v in enumerate(edu_counts.values):
    axes[1].text(i, v + 2, str(v), ha='center', color='white', fontsize=10)
axes[1].set_title('Candidate Breakdown by Degree', color='white')
axes[1].tick_params(colors='white')
axes[1].set_ylabel('Count', color='#94a3b8')

plt.tight_layout()
plt.show()

In [ ]:
# Figure 4: Shortlisting Acceptance Rate by Role and Degree
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
fig.suptitle('Figure 4: Acceptance Ratio by Role and Education', fontsize=13, fontweight='bold', color='white')

role_sl = df.groupby('Job_Role')['Shortlisted'].mean().mul(100).round(1).sort_values()
bars = axes[0].barh(role_sl.index, role_sl.values, color='#06b6d4')
for bar, v in zip(bars, role_sl.values):
    axes[0].text(v + 0.5, bar.get_y() + bar.get_height()/2, f'{v}%', va='center', color='white', fontsize=10)
axes[0].set_title('Shortlisting Rate by Job Role (%)', color='white')
axes[0].tick_params(colors='white')
axes[0].set_xlabel('Shortlisting Rate (%)', color='#94a3b8')

edu_sl = df.groupby('Education')['Shortlisted'].mean().mul(100).round(1).sort_values()
bars2 = axes[1].bar(edu_sl.index, edu_sl.values, color='#10b981', width=0.5)
for bar, v in zip(bars2, edu_sl.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, v + 0.5, f'{v}%', ha='center', color='white', fontsize=10)
axes[1].set_title('Shortlisting Rate by Degree (%)', color='white')
axes[1].tick_params(colors='white')
axes[1].set_ylabel('Shortlisting Rate (%)', color='#94a3b8')

plt.tight_layout()
plt.show()

In [ ]:
# Figure 5: Feature Interaction - Experience vs CGPA
fig, ax = plt.subplots(figsize=(9, 5))

sample = df.dropna(subset=['Experience_Years', 'CGPA']).copy()
shortlisted = sample[sample['Shortlisted'] == 1]
rejected    = sample[sample['Shortlisted'] == 0]

ax.scatter(rejected['Experience_Years'], rejected['CGPA'], c='#f87171', alpha=0.5, s=18, label='Rejected (0)')
ax.scatter(shortlisted['Experience_Years'], shortlisted['CGPA'], c='#4ade80', alpha=0.5, s=18, label='Shortlisted (1)')

ax.axvline(2.5, color='#facc15', linestyle='--', linewidth=1, alpha=0.7, label='Exp Threshold (2.5 yrs)')
ax.axhline(7.5, color='#818cf8', linestyle='--', linewidth=1, alpha=0.7, label='CGPA Threshold (7.5)')

ax.set_xlabel('Experience (Years)', color='#94a3b8')
ax.set_ylabel('CGPA', color='#94a3b8')
ax.set_title('Figure 5: Experience vs CGPA by Shortlisting Outcome', color='white', fontweight='bold')
ax.tick_params(colors='white')
ax.legend(fontsize=9, labelcolor='white')

plt.tight_layout()
plt.show()

## 6. Key Statistical Findings and Next Steps

In [ ]:
sl_rate = df['Shortlisted'].mean() * 100
avg_exp_sl = df[df['Shortlisted']==1]['Experience_Years'].mean()
avg_exp_rej = df[df['Shortlisted']==0]['Experience_Years'].mean()
avg_cgpa_sl = df[df['Shortlisted']==1]['CGPA'].mean()
avg_cgpa_rej = df[df['Shortlisted']==0]['CGPA'].mean()

print('====================================================')
print('           WEEK 1 STATISTICAL SUMMARY')
print('====================================================')
print(f'Shortlisting Rate          : {sl_rate:.1f}%')
print(f'Avg Experience (Shortlist) : {avg_exp_sl:.2f} years vs (Rejected): {avg_exp_rej:.2f} years')
print(f'Avg CGPA (Shortlist)       : {avg_cgpa_sl:.2f} vs (Rejected): {avg_cgpa_rej:.2f}')
print(f'Missing Values Identified  : {df.isnull().sum().sum()} across 4 attributes')
print(f'Duplicate Records          : {df.duplicated().sum()}')
print('====================================================')

### Week 1 Deliverables Summary

| Milestone Task | Status | Output Location |
|---|---|---|
| Dataset Generation Pipeline | Completed | `src/generate_data.py` |
| Raw Dataset Generation | Completed | `data/raw_resumes.csv` |
| Exploratory Data Analysis & Quality Audit | Completed | `notebooks/week1_data_collection_eda.ipynb` |
| Recruiter Dashboard Preview | Completed | `app.py` (Dataset Explorer) |

---

### Planned Tasks for Week 2
- Outlier detection study (IQR and Z-Score comparison).
- Missing value report and correlation matrix generation.
- Development of interactive EDA viewer in dashboard.